In [1]:
# Client Setup
from dotenv import load_dotenv
import voyageai

load_dotenv()

client = voyageai.Client()

d:\projetos\claude-api\.venv\Lib\site-packages\langchain_core\utils\pydantic.py:41: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1
d:\projetos\claude-api\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Chunk by section
import re


def chunk_by_section(document_text):
    pattern = r"\n## "
    return re.split(pattern, document_text)

In [3]:
# Embedding Generation
def generate_embedding(chunks, model="voyage-4-large", input_type="query"):
    is_list = isinstance(chunks, list)
    input = chunks if is_list else [chunks]
    result = client.embed(input, model=model, input_type=input_type)
    return result.embeddings if is_list else result.embeddings[0]

In [ ]:
# VectorIndex implementation
import math
from typing import Optional, Any, List, Dict, Tuple


class VectorIndex:
    def __init__(
        self,
        distance_metric: str = "cosine",
        embedding_fn=None,
    ):
        self.vectors: List[List[float]] = []
        self.documents: List[Dict[str, Any]] = []
        self._vector_dim: Optional[int] = None
        if distance_metric not in ["cosine", "euclidean"]:
            raise ValueError("distance_metric must be 'cosine' or 'euclidean'")
        self._distance_metric = distance_metric
        self._embedding_fn = embedding_fn

    def add_document(self, document: Dict[str, Any]):
        if not self._embedding_fn:
            raise ValueError(
                "Embedding function not provided during initialization."
            )
        if not isinstance(document, dict):
            raise TypeError("Document must be a dictionary.")
        if "content" not in document:
            raise ValueError(
                "Document dictionary must contain a 'content' key."
            )

        content = document["content"]
        if not isinstance(content, str):
            raise TypeError("Document 'content' must be a string.")

        vector = self._embedding_fn(content)
        self.add_vector(vector=vector, document=document)

    def search(
        self, query: Any, k: int = 1
    ) -> List[Tuple[Dict[str, Any], float]]:
        if not self.vectors:
            return []

        if isinstance(query, str):
            if not self._embedding_fn:
                raise ValueError(
                    "Embedding function not provided for string query."
                )
            query_vector = self._embedding_fn(query)
        elif isinstance(query, list) and all(
            isinstance(x, (int, float)) for x in query
        ):
            query_vector = query
        else:
            raise TypeError(
                "Query must be either a string or a list of numbers."
            )

        if self._vector_dim is None:
            'hello'
            return []

        if len(query_vector) != self._vector_dim:
            raise ValueError(
                f"Query vector dimension mismatch. Expected {self._vector_dim}, got {len(query_vector)}"
            )

        if k <= 0:
            raise ValueError("k must be a positive integer.")

        if self._distance_metric == "cosine":
            dist_func = self._cosine_distance
        else:
            dist_func = self._euclidean_distance

        distances = []
        for i, stored_vector in enumerate(self.vectors):
            distance = dist_func(query_vector, stored_vector)
            distances.append((distance, self.documents[i]))

        distances.sort(key=lambda item: item[0])

        return [(doc, dist) for dist, doc in distances[:k]]

    def add_vector(self, vector, document: Dict[str, Any]):
        if not isinstance(vector, list) or not all(
            isinstance(x, (int, float)) for x in vector
        ):
            raise TypeError("Vector must be a list of numbers.")
        if not isinstance(document, dict):
            raise TypeError("Document must be a dictionary.")
        if "content" not in document:
            raise ValueError(
                "Document dictionary must contain a 'content' key."
            )

        if not self.vectors:
            self._vector_dim = len(vector)
        elif len(vector) != self._vector_dim:
            raise ValueError(
                f"Inconsistent vector dimension. Expected {self._vector_dim}, got {len(vector)}"
            )

        self.vectors.append(list(vector))
        self.documents.append(document)

    def _euclidean_distance(
        self, vec1: List[float], vec2: List[float]
    ) -> float:
        if len(vec1) != len(vec2):
            raise ValueError("Vectors must have the same dimension")
        return math.sqrt(sum((p - q) ** 2 for p, q in zip(vec1, vec2)))

    def _dot_product(self, vec1: List[float], vec2: List[float]) -> float:
        if len(vec1) != len(vec2):
            raise ValueError("Vectors must have the same dimension")
        return sum(p * q for p, q in zip(vec1, vec2))

    def _magnitude(self, vec: List[float]) -> float:
        return math.sqrt(sum(x * x for x in vec))

    def _cosine_distance(self, vec1: List[float], vec2: List[float]) -> float:
        if len(vec1) != len(vec2):
            raise ValueError("Vectors must have the same dimension")

        mag1 = self._magnitude(vec1)
        mag2 = self._magnitude(vec2)

        if mag1 == 0 and mag2 == 0:
            return 0.0
        elif mag1 == 0 or mag2 == 0:
            return 1.0

        dot_prod = self._dot_product(vec1, vec2)
        cosine_similarity = dot_prod / (mag1 * mag2)
        cosine_similarity = max(-1.0, min(1.0, cosine_similarity))

        return 1.0 - cosine_similarity

    def __len__(self) -> int:
        return len(self.vectors)

    def __repr__(self) -> str:
        has_embed_fn = "Yes" if self._embedding_fn else "No"
        return f"VectorIndex(count={len(self)}, dim={self._vector_dim}, metric='{self._distance_metric}', has_embedding_fn='{has_embed_fn}')"

In [5]:
with open("./report.md", "r") as f:
    text = f.read()

In [9]:
# 1. Chunk the text by section
chunks = chunk_by_section(text)

In [10]:
# 2. Generate embeddings for each chunk
embeddings = generate_embedding(chunks)

In [12]:
len(embeddings) == len(chunks)

True

In [27]:
# 3. Create a vector store and add each embedding to it
# Note: converted to a bulk operation to avoid rate limiting errors from VoyageAI
# store = VectorIndex(embedding_fn=generate_embedding)
# (store.add_document({"content": chunk}) for chunk in chunks)


store = VectorIndex()

for embedding, chunk in zip(embeddings, chunks):
    store.add_vector(embedding, {"content": chunk})

In [28]:
store.vectors

[[-0.03839186578989029,
  0.00018836009257938713,
  0.013744288124144077,
  0.02518506348133087,
  -0.04729877784848213,
  -0.010058669373393059,
  -0.02288155071437359,
  0.011517559178173542,
  -0.04607023671269417,
  -0.01074972189962864,
  0.025799334049224854,
  0.034706246107816696,
  -0.05467001721262932,
  0.02902424894273281,
  -0.003935166168957949,
  -0.031020626425743103,
  0.01051937136799097,
  -0.037470459938049316,
  0.00483737513422966,
  -0.04944872111082077,
  0.031634896993637085,
  -0.03700975701212883,
  -0.012746099382638931,
  -0.008177466690540314,
  0.0037432070821523666,
  -0.017506690695881844,
  0.024724360555410385,
  0.010365803726017475,
  -0.014281773008406162,
  -0.04514883831143379,
  0.028256414458155632,
  0.033324141055345535,
  0.020117338746786118,
  -0.05467001721262932,
  0.02457079477608204,
  -0.03209559991955757,
  -0.00424230145290494,
  -0.016738854348659515,
  -0.01243896409869194,
  -0.007563197985291481,
  0.03869900107383728,
  -0.0013

In [29]:
store._vector_dim

1024

In [17]:
# 4. Some time later, a user will ask a question. Generate an embedding for it
user_embedding = generate_embedding("What did the software engineering dept do last year?")

In [30]:
# 5. Search the store with the embedding, find the 2 most relevant chunks
results = store.search(user_embedding, k=2)

for doc, dist in results:
    print(f"Distance: {dist:.2f}")
    print(f"Document: {doc['content'][:200]}")
    print("\n")


Distance: 0.72
Document: Executive Summary

This report synthesizes the key findings and ongoing research efforts across the organization's diverse operational and R&D departments for the past fiscal year. Our strength lies i


Distance: 0.75
Document: Methodology

The insights compiled within this Annual Interdisciplinary Research Review represent a synthesis of findings drawn from standard departmental reporting cycles, specialized project updates




In [22]:
user_embedding

[0.021580491214990616,
 -0.04440302774310112,
 -0.03400091826915741,
 0.02577238529920578,
 0.022667279466986656,
 0.012109916657209396,
 -0.0027945961337536573,
 -0.005317495204508305,
 -0.017699109390378,
 0.004851729609072208,
 0.06272315979003906,
 0.02359881065785885,
 -0.038969092071056366,
 -0.026393409818410873,
 0.003202141262590885,
 0.02701442874968052,
 0.0028916308656334877,
 -0.01397298090159893,
 -5.7311055570608005e-05,
 -0.025306621566414833,
 -0.003648500656709075,
 0.04564506933093071,
 0.00045606258208863437,
 0.024996109306812286,
 -0.0009363837889395654,
 -0.013662470504641533,
 0.0040366388857364655,
 0.008655485697090626,
 -0.006481910590082407,
 -0.010013969615101814,
 -0.00138759461697191,
 0.017000459134578705,
 0.002833409933373332,
 0.03415617346763611,
 -0.007258187048137188,
 -0.026238152757287025,
 -0.04285047575831413,
 -0.05123426020145416,
 -0.07700665295124054,
 0.020648960024118423,
 -0.005744447465986013,
 -0.02794596180319786,
 -0.0166899487376213